# 07.02 · Similarity search: brute force to ANN

**Chapter:** Embeddings & Vector Search · **Milestone:** M3

## Goals
- [ ] Exact (brute-force) search with NumPy and its cost
- [ ] Approximate nearest neighbour (ANN) indexes: HNSW, IVF, PQ — with FAISS
- [ ] Measure the trade-off: recall@k vs latency vs memory

## Resources
- [Pinecone Learn – Faiss: the missing manual](https://www.pinecone.io/learn/series/faiss/) — free
- [FAISS wiki](https://github.com/facebookresearch/faiss/wiki) — free
- [ANN-Benchmarks](https://ann-benchmarks.com/) — free

Every vector database (07.03) is an ANN index + storage + filtering + an API. Knowing the index types lets you tune any of them.

In [ ]:
import time

import faiss
import numpy as np

from aieng import rag

## 1. Real data: the mini-Wikipedia corpus
3,200 passages embedded with bge-small (cached in `data/` after the first run — `aieng.rag`, packaged from 08.01).

In [ ]:
docs, X = rag.corpus(), rag.index()
print(X.shape, X.dtype, f"{X.nbytes / 1e6:.1f} MB")
for pid, score, text in rag.search("When was Abraham Lincoln born?", k=3):
    print(f"{score:.3f} [{pid}] {text[:100]}")

## 2. Brute force: exact, simple, O(N·d) per query
For up to ~100k–1M vectors, brute force on a laptop is often fast enough. Don't add infrastructure you don't need.

In [ ]:
def brute_force(Q, X, k=10):
    scores = Q @ X.T  # (n_queries, N)
    idx = np.argpartition(-scores, k, axis=1)[:, :k]  # top-k unsorted in O(N)
    order = np.take_along_axis(scores, idx, axis=1).argsort(axis=1)[:, ::-1]
    return np.take_along_axis(idx, order, axis=1)

# Scale up to 200k vectors: real embeddings + small noise (keeps the cluster structure of real data;
# pure random vectors have none and make every ANN index look terrible). Queries = real embedded eval questions.
rng = np.random.default_rng(0)
d = X.shape[1]
N = 200_000
Xbig = X[rng.integers(0, len(X), N)] + 0.03 * rng.normal(size=(N, d)).astype("float32")
Xbig /= np.linalg.norm(Xbig, axis=1, keepdims=True)
Q = rag.embed([r["question"] for r in rag.eval_set()[:200]], query=True).astype("float32")

start = time.perf_counter(); truth = brute_force(Q, Xbig); bf_ms = (time.perf_counter() - start) * 1000 / len(Q)
print(f"brute force over {N:,} vectors: {bf_ms:.2f} ms/query, memory {Xbig.nbytes / 1e6:.0f} MB")

## 3. FAISS indexes
| Index | Idea | Trade-off |
|---|---|---|
| `IndexFlatIP` | brute force, optimised (SIMD/BLAS) | exact; O(N) |
| **HNSW** | navigable small-world graph: greedy walk from coarse to fine layers | excellent recall/speed; more memory; slow-ish build. **Default in most vector DBs.** |
| **IVF** | k-means into `nlist` cells; search only `nprobe` nearest cells | fast; recall depends on `nprobe` |
| **PQ** | compress vectors into small codes (e.g. 384 floats → 48 bytes) | huge memory savings, lower recall |

Inner product (`IP`) on normalised vectors = cosine similarity.

In [ ]:
def recall_at_k(pred, truth, k=10):
    return np.mean([len(set(p[:k]) & set(t[:k])) / k for p, t in zip(pred, truth)])


def bench(name, index, build=True, **params):
    start = time.perf_counter()
    if build:
        if not index.is_trained:
            index.train(Xbig)
        index.add(Xbig)
    build_s = time.perf_counter() - start
    for key, val in params.items():
        faiss.ParameterSpace().set_index_parameter(index, key, val)
    start = time.perf_counter(); _, I = index.search(Q, 10); ms = (time.perf_counter() - start) * 1000 / len(Q)
    print(f"{name:28} build {build_s:5.1f}s | {ms:6.3f} ms/query | recall@10 {recall_at_k(I, truth):.3f}")
    return index


bench("Flat (exact)", faiss.IndexFlatIP(d))
hnsw = bench("HNSW M=32 efSearch=16", faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT), efSearch=16)
bench("HNSW M=32 efSearch=128", hnsw, build=False, efSearch=128)
ivf = bench("IVF1024 nprobe=1", faiss.index_factory(d, "IVF1024,Flat", faiss.METRIC_INNER_PRODUCT), nprobe=1)
bench("IVF1024 nprobe=32", ivf, build=False, nprobe=32)
bench("IVF1024,PQ48 nprobe=32", faiss.index_factory(d, "IVF1024,PQ48", faiss.METRIC_INNER_PRODUCT), nprobe=32)

Notes:
- Recall here is measured against exact search, not against relevance: it's "how much of the true top-10 did the index find".
- The knobs you'll see in every vector DB: HNSW `M` / `ef_construction` / `ef_search`, IVF `nlist` / `nprobe`, quantisation type.
- Memory: HNSW adds graph links (~M × 8 bytes per vector); PQ48 stores 48 bytes per vector instead of 1,536.

## 4. Save / load

In [ ]:
from aieng import DATA

path = str(DATA / "mini-wiki-hnsw.faiss")
small = faiss.IndexHNSWFlat(X.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
small.add(X)
faiss.write_index(small, path)
reloaded = faiss.read_index(path)
_, I = reloaded.search(rag.embed(["When was Abraham Lincoln born?"], query=True), 3)
print([docs[i][:80] for i in I[0]])

## Exercises
1. Plot recall@10 vs ms/query for HNSW with `efSearch` in [8, 16, 32, 64, 128, 256] and IVF with `nprobe` in [1, 2, 4, …, 128].
2. Rerun the benchmark with **pure random** vectors (`rng.normal`). Why does every ANN index collapse? (Hint: in high dimensions, random points are all nearly equidistant.)
3. Try `faiss.IndexBinaryFlat` on sign-binarised vectors (`X > 0`) with Hamming distance: 32× less memory. Rerank the top-100 with full vectors. Recall?

In [ ]:
# your code here